# Rocket Roll Dynamics
Based on [RocketPy Roll Equations](https://docs.rocketpy.org/en/v1.0.0/technical/aerodynamics/roll_equations.html)

## Theory

Roll motion is driven by aerodynamic asymmetries — primarily fin cant angle $\delta$. The net roll moment is:

$$M_{roll} = M_{forcing} - M_{damping}$$

### Roll Forcing Moment
Each canted fin produces a side force. With $N$ fins:

$$M_f = N\,(Y_{MA} + r_t)\,(C_{N\alpha})_1\,\delta\,\bar{q}\,A_r$$

Expressed as a coefficient:
$$C_{lf} = \frac{N\,(Y_{MA} + r_t)\,(C_{N\alpha})_1}{L_r}\,\delta$$

### Roll Damping Moment
Fins moving through air due to rotation experience restoring forces. The damping coefficient (trapezoidal fins):

$$C_{ld} = \frac{2N\,(C_{N\alpha})_1}{A_r\,L_r^2}\,\cos(\delta)\int_{r_t}^{s+r_t} c(\xi)\,\xi^2\,d\xi$$

where for trapezoidal fins:

$$\int_{r_t}^{s+r_t} c(\xi)\,\xi^2\,d\xi = \frac{s}{12}\left[(C_r+3C_t)s^2 + 4(C_r+2C_t)s\,r_t + 6(C_r+C_t)r_t^2\right]$$

The damping moment itself:
$$M_d = C_{ld}\,\frac{\omega\,L_r}{2v_0}\,\bar{q}\,A_r\,L_r$$

### Equation of Motion

$$I_{xx}\,\dot{\omega} = (C_{lf} - C_{ld}\,\frac{\omega\,L_r}{2v_0})\,\bar{q}\,A_r\,L_r$$

**Steady-state roll rate** (when $\dot{\omega} = 0$):

$$\omega_{ss} = \frac{C_{lf\delta}\,\delta}{C_{ld\omega}}\,\frac{2v_0}{L_r}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

## Rocket & Fin Parameters

| Symbol | Description | Unit |
|--------|-------------|------|
| $N$ | Number of fins | — |
| $r_t$ | Tube radius at fin root | m |
| $s$ | Fin semi-span | m |
| $C_r$ | Root chord | m |
| $C_t$ | Tip chord | m |
| $\delta$ | Fin cant angle | rad |
| $L_r$ | Reference length (diameter) | m |
| $A_r$ | Reference area | m² |
| $I_{xx}$ | Roll moment of inertia | kg·m² |
| $(C_{N\alpha})_1$ | Normal force slope per fin | 1/rad |

In [ ]:
# ── Rocket geometry ──────────────────────────────────────────────────────────
N    = 4            # number of fins
r_t  = 0.038        # tube radius at fin root [m]  (76mm diameter)
s    = 0.080        # fin semi-span [m]
C_r  = 0.120        # root chord [m]
C_t  = 0.060        # tip chord [m]

L_r  = 2 * r_t      # reference length = diameter [m]
A_r  = np.pi * r_t**2  # reference area [m²]

# ── Fin aerodynamics ──────────────────────────────────────────────────────────
# Normal force gradient (per fin, per radian) — Barrowman approximation
AR_fin = 2 * s**2 / (0.5 * (C_r + C_t) * s)   # fin aspect ratio
CNa1 = 2 * np.pi * AR_fin / (2 + np.sqrt(4 + AR_fin**2))
print(f"Fin aspect ratio: {AR_fin:.3f}")
print(f"(CNα)₁ = {CNa1:.4f} /rad")

# Mean aerodynamic chord spanwise center Y_MA (trapezoidal)
Y_MA = s / 3 * (C_r + 2*C_t) / (C_r + C_t)   # [m] from fin root
print(f"Y_MA  = {Y_MA*1000:.2f} mm from fin root")

# ── Roll / inertia ────────────────────────────────────────────────────────────
delta_deg = 2.0           # fin cant angle [degrees]
delta = np.radians(delta_deg)

I_xx = 0.015              # roll moment of inertia [kg·m²]

In [ ]:
# ── Trapezoidal fin integral  ∫ c(ξ) ξ² dξ from r_t to s+r_t ─────────────────
def fin_integral_trap(C_r, C_t, s, r_t):
    """Closed-form integral for trapezoidal fins (RocketPy eq.)."""
    return s / 12 * (
        (C_r + 3*C_t) * s**2
        + 4*(C_r + 2*C_t) * s * r_t
        + 6*(C_r + C_t)   * r_t**2
    )

I_fin = fin_integral_trap(C_r, C_t, s, r_t)
print(f"Fin integral = {I_fin:.6f} m⁴")

# ── Roll coefficients ─────────────────────────────────────────────────────────
# Forcing coefficient slope  Clf_delta = Clf / delta
Clf_delta = N * (Y_MA + r_t) * CNa1 / L_r

# Damping coefficient  Cld (multiplied by ωLr/2v0 to get moment coefficient)
Cld = (2 * N * CNa1 * np.cos(delta)) / (A_r * L_r**2) * I_fin

print(f"Clf_delta = {Clf_delta:.4f} /rad")
print(f"Cld       = {Cld:.4f} /rad")

## Steady-State Roll Rate vs. Velocity

In [ ]:
v0_range = np.linspace(10, 300, 500)   # rocket speed [m/s]

# ω_ss = (Clf_delta * δ) / Cld  *  2*v0 / L_r
omega_ss = (Clf_delta * delta) / Cld * (2 * v0_range / L_r)
rpm_ss   = omega_ss * 60 / (2 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(v0_range, omega_ss, 'b')
axes[0].set_xlabel('Rocket speed  $v_0$  [m/s]')
axes[0].set_ylabel('Steady-state roll rate  $\\omega_{ss}$  [rad/s]')
axes[0].set_title(f'Steady-State Roll Rate  (δ = {delta_deg}°)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(v0_range, rpm_ss, 'r')
axes[1].set_xlabel('Rocket speed  $v_0$  [m/s]')
axes[1].set_ylabel('Steady-state roll rate  [RPM]')
axes[1].set_title(f'Steady-State Roll Rate  (δ = {delta_deg}°)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Time-Domain Roll Simulation

Integrating $I_{xx}\dot{\omega} = (C_{lf} - C_{ld}\,\frac{\omega L_r}{2v_0})\,\bar{q}\,A_r\,L_r$ with a velocity–time profile.

In [ ]:
# ── Simple velocity profile (burn + coast) ────────────────────────────────────
rho   = 1.225          # air density [kg/m³]  (sea level)
t_end = 10.0           # simulation time [s]
dt    = 0.005

# Piecewise velocity: accelerate during burn, then coast to apogee
t_burn  = 2.5          # burn time [s]
v_max   = 200.0        # max speed at burnout [m/s]

def v0_profile(t):
    """Simple triangular-ish velocity profile."""
    if t <= t_burn:
        return v_max * (t / t_burn)
    else:
        return v_max * np.exp(-0.05 * (t - t_burn))  # gentle coast deceleration

v0_vec = np.vectorize(v0_profile)

# ── ODE ───────────────────────────────────────────────────────────────────────
def roll_ode(t, y):
    omega, phi = y
    v0   = v0_profile(t)
    q_bar = 0.5 * rho * v0**2

    # Moments
    M_forcing = Clf_delta * delta * q_bar * A_r * L_r
    M_damping = Cld * (omega * L_r / (2 * v0 + 1e-9)) * q_bar * A_r * L_r

    domega = (M_forcing - M_damping) / I_xx
    dphi   = omega
    return [domega, dphi]

t_span = (0, t_end)
t_eval = np.arange(0, t_end, dt)
sol = solve_ivp(roll_ode, t_span, [0.0, 0.0], t_eval=t_eval, max_step=dt, method='RK45')

omega_t  = sol.y[0]
phi_t    = np.degrees(sol.y[1])
rpm_t    = omega_t * 60 / (2 * np.pi)
t        = sol.t
v_t      = v0_vec(t)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(t, v_t, 'k')
axes[0].set_ylabel('Speed  [m/s]')
axes[0].set_title('Rocket Velocity Profile')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(t_burn, color='gray', ls='--', label='Burnout')
axes[0].legend()

axes[1].plot(t, rpm_t, 'b')
axes[1].set_ylabel('Roll rate  [RPM]')
axes[1].set_title(f'Roll Rate  (δ = {delta_deg}°,  I_xx = {I_xx} kg·m²)')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(t_burn, color='gray', ls='--')

axes[2].plot(t, phi_t % 360, 'r')
axes[2].set_xlabel('Time  [s]')
axes[2].set_ylabel('Roll angle  [deg]')
axes[2].set_title('Roll Angle (mod 360°)')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(t_burn, color='gray', ls='--')

plt.tight_layout()
plt.show()

print(f"Peak roll rate: {rpm_t.max():.1f} RPM at t = {t[np.argmax(rpm_t)]:.2f} s")

## Parametric Study: Cant Angle Effect

In [ ]:
cant_angles = [0.5, 1.0, 2.0, 3.0, 5.0]   # degrees

fig, ax = plt.subplots(figsize=(10, 5))

for d_deg in cant_angles:
    d = np.radians(d_deg)
    Clf_d = N * (Y_MA + r_t) * CNa1 / L_r   # same geometry
    Cld_d = (2 * N * CNa1 * np.cos(d)) / (A_r * L_r**2) * I_fin

    def ode_cant(t, y, d=d, Clf_d=Clf_d, Cld_d=Cld_d):
        omega, phi = y
        v0    = v0_profile(t)
        q_bar = 0.5 * rho * v0**2
        Mf = Clf_d * d * q_bar * A_r * L_r
        Md = Cld_d * (omega * L_r / (2 * v0 + 1e-9)) * q_bar * A_r * L_r
        return [(Mf - Md) / I_xx, omega]

    sol_c = solve_ivp(ode_cant, t_span, [0.0, 0.0], t_eval=t_eval, max_step=dt, method='RK45')
    ax.plot(sol_c.t, sol_c.y[0] * 60 / (2*np.pi), label=f'δ = {d_deg}°')

ax.axvline(t_burn, color='gray', ls='--', label='Burnout')
ax.set_xlabel('Time  [s]')
ax.set_ylabel('Roll rate  [RPM]')
ax.set_title('Roll Rate vs. Cant Angle')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Parameter | Value |
|-----------|-------|
| Fin cant angle δ | `delta_deg` ° |
| $(C_{N\alpha})_1$ | `CNa1` /rad |
| $C_{lf\delta}$ | `Clf_delta` /rad |
| $C_{ld}$ | `Cld` /rad |

Steady-state roll rate scales **linearly with velocity** and **linearly with cant angle** (for small $\delta$).  
Time to reach steady-state is governed by $\tau = I_{xx} / (C_{ld}\,\bar{q}\,A_r\,L_r^2 / (2v_0))$.